In [2]:
%matplotlib widget
import qiskit
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt
import numpy as np
import circuit_lib as lib
import importlib as imp

print("qiskit version: ", qiskit.version.get_version_info())

qiskit version:  2.0.0


In [ ]:
# --------------------------------------------------------
# --- Example: computing a statevetor ---
# --------------------------------------------------------
imp.reload(lib)
def create_circuit():
    # initialize a quantum circuit
    a = qiskit.QuantumRegister(1, "a")
    qc = qiskit.QuantumCircuit(a,name="A")

    # let's assume that |1> is a 'good' state (and it has low probability):
    coef_v = 0.1
    qc.ry(2*np.arccos(coef_v), a)
    qc.x(a)

    # store the circuit as a gate:
    gA = qc.to_gate()

    # draw the circuit
    print(qc.draw(fold = 110))

    return qc, gA
# ---------------------------------------------------------------
def compute_state_vector(circ):
    # initialize the Aer simulator using the constructed circuit
    simulator = AerSimulator()
    qc_stv = circ.copy()
    qc_stv.save_statevector()
    transpiled_qc = qiskit.transpile(qc_stv, simulator)

    # run the simulator
    res_sim = simulator.run(transpiled_qc).result()

    # get the statevector
    statevec = res_sim.data()['statevector']
    statevec = np.asarray(statevec)
    print("Statevector:")
    lib.print_array(statevec)
    return statevec
# ---------------------------------------------------------------
def analyse_good_bad_states(circ, statevec = None, flag_print_opt_number = False):
    if statevec is not None:
        str_wv, pp, qq = lib.Wavefunction_adv(statevec, column=True, systems=[1], width=10)
    else:
        str_wv, pp, qq = lib.Wavefunction_adv(circ, column=True, systems=[1], width=10)

    print("state amplitudes:")
    print(str_wv)

    lib.analysis_prob(pp, qq, flag_print_opt_number)
    return
# ---------------------------------------------------------------
circ, circ_as_gate = create_circuit()
statevec = compute_state_vector(circ)

print("\n---Analysis ---")
analyse_good_bad_states(None, statevec, True)

   ┌────────────┐┌───┐
a: ┤ Ry(2.9413) ├┤ X ├
   └────────────┘└───┘
Statevector:
 0.995+0.000j  0.100+0.000j 

---Analysis ---
state amplitudes:
        0.995+0.000j |0>   
        0.100+0.000j |1>   

prob. of the good state: 1.000e-02
prob. of the bad state: 9.900e-01
prob. of bad state + good state: 1.000e+00
the optimal number of amplifications:  7.840854384487761


In [3]:
# -----------------------------------------
# --- Amplitude amplification circuit ---
# -----------------------------------------
def construct_amplification_circuit_one_round(A):
    """
    A is the preparation circuit whose 'good' should be amplified;
    A should be given as a gate.

    Let's assume that |1> is the 'good' state.
    
    One round (iteration) of amplitude amplification is
        Q = - A R_0 A^{-1} R_good
    where 
       R_0    is the reflection w.r.t the initial state,
       R_good is the reflection w.r.t the 'good' state.

    return: Q as a gate.
    """

    reg_a = qiskit.QuantumRegister(1, "a")
    Q     = qiskit.QuantumCircuit(reg_a,name="AA")

    # change the sign of the good state
    Q.z(reg_a)

    # add an inverse preparation state
    gAi = A.inverse()
    Q.append(gAi, reg_a)

    # change the sign of the initial state, i.e. the zero state
    Q.x(reg_a)
    Q.z(reg_a)
    Q.x(reg_a)

    # add the preparation creation gate again:
    Q.append(A, reg_a)

    # draw the circuit
    print(Q.draw())

    # convert the circuit to a gate
    gQ = Q.to_gate()
    return gQ
# ----------------------------------------------
one_AA_iter_as_gate = construct_amplification_circuit_one_round(circ_as_gate)

   ┌───┐┌──────┐┌───┐┌───┐┌───┐┌───┐
a: ┤ Z ├┤ A_dg ├┤ X ├┤ Z ├┤ X ├┤ A ├
   └───┘└──────┘└───┘└───┘└───┘└───┘


In [5]:
# --------------------------------------------------
# --- Several amplitude-amplification iterations ---
# --------------------------------------------------
def amplitude_amplification(A, Q, n_iter, flag_step_by_step):
    a = qiskit.QuantumRegister(1, "a")
    qc = qiskit.QuantumCircuit(a,name="AA")
    qc.append(A, a)

    print("--- before amplification ---")
    print(qc.draw(fold = 110))
    str_wv, pp, qq = lib.Wavefunction_adv(qc, column=True, systems=[1], width=10)
    print(str_wv)
    lib.analysis_prob(pp, qq)
    if flag_step_by_step:
        # --- show the statevector after each AA iteration ---
        for i_iter in range(n_iter):
            print("\n\n--------------------------------------------------------------")
            print("--- after the {:d}-th amplification ---".format(i_iter+1))
            qc.append(Q, a)
            print(qc.draw(fold = 110))
            analyse_good_bad_states(qc)
    else:
        # --- show the statevector after n_iter AA iterations ---
        print("\n\n--------------------------------------------------------------")
        print("--- after {:d} amplifications ---".format(n_iter))
        for i_iter in range(n_iter):
            qc.append(Q, a)
        print(qc.draw(fold = 110))
        analyse_good_bad_states(qc)
    return
# ----------------------------------------------
amplitude_amplification(
    A = circ_as_gate,
    Q = one_AA_iter_as_gate, 
    n_iter = 8, 
    flag_step_by_step = True
)

--- before amplification ---
   ┌───┐
a: ┤ A ├
   └───┘
        0.995+0.000j |0>   
        0.100+0.000j |1>   

prob. of the good state: 1.000e-02
prob. of the bad state: 9.900e-01
prob. of bad state + good state: 1.000e+00
the optimal number of amplifications:  7.840854384487761


--------------------------------------------------------------
--- after the 1-th amplification ---
   ┌───┐┌────┐
a: ┤ A ├┤ AA ├
   └───┘└────┘
state amplitudes:
       -0.955+0.000j |0>   
       -0.296+0.000j |1>   

prob. of the good state: 8.762e-02
prob. of the bad state: 9.124e-01
prob. of bad state + good state: 1.000e+00


--------------------------------------------------------------
--- after the 2-th amplification ---
   ┌───┐┌────┐┌────┐
a: ┤ A ├┤ AA ├┤ AA ├
   └───┘└────┘└────┘
state amplitudes:
        0.877+0.000j |0>   
        0.480+0.000j |1>   

prob. of the good state: 2.306e-01
prob. of the bad state: 7.694e-01
prob. of bad state + good state: 1.000e+00


------------------------------